### Fix issue with 121 dataset
- creating the 121 dataset with a merge from MMSE_Category from np_with_MMSE as there was a mistake when creating the previous dataset

In [0]:
%sql
ALTER TABLE np_121_binary SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name'); ALTER TABLE np_121_binary DROP COLUMN MMSE_Category;
CREATE OR REPLACE TABLE np_121_binary AS
SELECT 
    a.*,
    b.MMSE_Category
FROM np_121_binary a
LEFT JOIN (
    SELECT DISTINCT PTID, MMSE_Category
    FROM np_with_mmse
) b
ON a.PTID = b.PTID;

In [0]:
%sql
SELECT COUNT(*) FROM np_121_binary;

In [0]:
%sql
SELECT COUNT(MMSE_Category) FROM np_121_binary;

In [0]:
%sql
SELECT * FROM np_121_binary

### Drop Null column 'y_mmse_binary' and replace with fixed values

In [0]:
%sql
ALTER TABLE np_121_binary_fixed RENAME COLUMN NPIG_x TO NPIG;
CREATE OR REPLACE TABLE np_121_binary_fixed AS
  SELECT *,
    CASE 
      WHEN MMSE_Category IN (
        'Mild AD',
        'Moderate AD',
        'Severe AD'
      ) THEN 1
      WHEN MMSE_Category IN (
        'Normal cognition (Fully intact)',
        'Normal cognition / Possible MCI (Borderline)'
      ) THEN 0
      WHEN MMSE_Category = 'Invalid Score' THEN NULL  
      ELSE NULL 
  END AS y_mmse_binary
FROM np_121_binary;
SELECT * FROM np_121_binary_fixed

Rename to have variable match other table

In [0]:
%sql
ALTER TABLE np_121_binary_fixed SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');
ALTER TABLE np_121_binary_fixed RENAME COLUMN NPIG_x TO NPIG;

In [0]:
%sql
SELECT WORD1DL, WORD3DL, WORD2DL, MMDRAW, MMW, MMO, MMREPEAT, MMR, MMHAND, VSWEIGHT, GDBORED, VSHEIGHT, VSTMPSRC, VSTEMP, NPIG, VSPULSE, VSBPSYS, GDTOTAL, y_mmse_binary
FROM np_121_binary_fixed

In [0]:
top_features = [
    "WORD1DL", "WORD3DL", "WORD2DL", "MMDRAW", "MMW",
    "MMLTR5_W", "MMO", "MMREPEAT", "MMR", "MMHAND",
    "VSWEIGHT", "GDBORED", "VSHEIGHT", "VSTMPSRC",
    "VSTEMP", "NPIG", "VSPULSE", "VSBPSYS", "GDTOTAL"
]

from pyspark.sql import functions as F

table_name = "NP_121_binary_fixed"
df = spark.table(table_name)

# keep only rows with valid label
df_model = df.filter(F.col("y_mmse_binary").isNotNull())
columns_to_keep = top_features + ["y_mmse_binary"]
columns_to_keep = [c for c in columns_to_keep if c in df_model.columns]

# KEEP ONLY these columns
df_model = df_model.select(*columns_to_keep)

In [0]:
display(df_model.select("y_mmse_binary").groupBy("y_mmse_binary").count())
for col in df_model.columns:
    print(col)

In [0]:
df_model.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("NP_Binary_SHAP")

In [0]:
%sql
SELECT * FROM np_binary_shap

Make it 50/50

In [0]:
import pandas as pd
df_from_spark = spark.table("NP_Binary_SHAP")
pdf = df_from_spark.toPandas()

In [0]:
import pandas as pd
# Separate minority and majority classes
df_1 = pdf[pdf['y_mmse_binary'] == 1]
df_0 = pdf[pdf['y_mmse_binary'] == 0]
 
# Randomly sample from majority class
df_0_sampled = df_0.sample(n=len(df_1), random_state=42)
 
# Combine and shuffle
balanced_df = (
    pd.concat([df_1, df_0_sampled])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
 
print(balanced_df['y_mmse_binary'].value_counts())
pdf = balanced_df

In [0]:
# make it a spark dataframe
spark_df = spark.createDataFrame(pdf)

# overwrite from spark to delta table
spark_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("NP_Binary_SHAP_50_50")

In [0]:
%sql
SELECT * FROM np_binary_shap_50_50